<a href="https://colab.research.google.com/github/Levan-Danelia/FRTB/blob/main/FRTB_EQCV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# --- Step 2 & 3: Establish Gross and Net Positions ---

# Create a DataFrame with the gross positions for the equity portfolio.
gross_data = {
    'Bucket': [6, 6, 6, 5, 5],
    'Issuer': ['Issuer A', 'Issuer B', 'Issuer B', 'Issuer C', 'Issuer D'],
    'CVR+': [-6447, -49220, 6200, 8990, 3638],
    'CVR-': [8842, 48581, -48309, -72303, -28347]
}
gross_df = pd.DataFrame(gross_data)

print("--- Step 2: Gross Positions ---")
print(gross_df)
print("\n" + "="*50 + "\n")

# Per Article 325g(2), we net positions for the same risk factor (by Issuer).
net_df = gross_df.groupby(['Bucket', 'Issuer']).sum().reset_index()

print("--- Step 3: Net Positions ---")
print("Netting the two 'Issuer B' positions as they belong to the same risk factor.")
print(net_df)
print("\n" + "="*50 + "\n")


# --- Step 4: Determine Correlation Parameters (Medium Scenario) ---

# Intra-bucket delta correlation for Buckets 5 & 6 - Article 325aq(2)(b)
delta_rho_kl = 0.25

# Cross-bucket delta correlation for Bucket 5 vs Bucket 6 - Article 325ar(a)
delta_gamma_bc = 0.15

# Curvature correlations are the square of delta correlations - Article 325ay(5)
# This is the BASELINE (Medium) curvature correlation.
curvature_rho_kl_medium = delta_rho_kl**2
curvature_gamma_bc_medium = delta_gamma_bc**2

print("--- Step 4: Correlation Parameters (Medium Scenario) ---")
print(f"Intra-Bucket Delta Correlation (rho_kl): {delta_rho_kl:.2%}")
print(f"Intra-Bucket Curvature Correlation (rho_kl^2): {curvature_rho_kl_medium:.2%}\n")
print(f"Cross-Bucket Delta Correlation (gamma_bc): {delta_gamma_bc:.2%}")
print(f"Cross-Bucket Curvature Correlation (gamma_bc^2): {curvature_gamma_bc_medium:.2%}")
print("\n" + "="*50 + "\n")


# --- Step 5: Calculate Bucket-Level Capital (Medium Scenario) ---

def calculate_bucket_capital(cvr_data, intra_bucket_corr):
    """Calculates Kb+, Kb-, and the final Kb for a given bucket."""

    # Safeguard function psi from Article 325g(4, 6)
    def psi(x, y):
        return 0 if x < 0 and y < 0 else 1

    # Isolate CVR+ and CVR- values for the bucket
    cvr_plus = cvr_data['CVR+'].values
    cvr_minus = cvr_data['CVR-'].values

    # Calculate Kb+ (Upward Scenario)
    sum_max_cvr_plus_sq = np.sum(np.maximum(cvr_plus, 0)**2)
    correlation_term_plus = 0
    if len(cvr_plus) > 1:
        # Simplified for 2 risk factors
        correlation_term_plus = 2 * intra_bucket_corr * cvr_plus[0] * cvr_plus[1] * psi(cvr_plus[0], cvr_plus[1])
    kb_plus = np.sqrt(max(0, sum_max_cvr_plus_sq + correlation_term_plus))

    # Calculate Kb- (Downward Scenario)
    sum_max_cvr_minus_sq = np.sum(np.maximum(cvr_minus, 0)**2)
    correlation_term_minus = 0
    if len(cvr_minus) > 1:
        correlation_term_minus = 2 * intra_bucket_corr * cvr_minus[0] * cvr_minus[1] * psi(cvr_minus[0], cvr_minus[1])
    kb_minus = np.sqrt(max(0, sum_max_cvr_minus_sq + correlation_term_minus))

    # Determine final Kb and selected scenario
    if kb_plus > kb_minus:
        kb_final = kb_plus
        scenario = "Upward"
    elif kb_minus > kb_plus:
        kb_final = kb_minus
        scenario = "Downward"
    else: # If they are equal, choose based on the larger sum of CVRs
        kb_final = kb_plus
        scenario = "Upward" if np.sum(cvr_plus) >= np.sum(cvr_minus) else "Downward"

    return kb_plus, kb_minus, kb_final, scenario

print("--- Step 5: Bucket-Level Capital (Medium Scenario) ---")

# Bucket 6 (Issuers A & B)
b6_data = net_df[net_df['Bucket'] == 6]
kb6_plus, kb6_minus, kb6_final, scenario6 = calculate_bucket_capital(b6_data, curvature_rho_kl_medium)
print("Bucket 6 (Issuers A & B):")
print(f"  - K_b+ = {kb6_plus:,.0f}")
print(f"  - K_b- = {kb6_minus:,.0f}")
print(f"  - Final K_b = {kb6_final:,.0f} (Selected Scenario: {scenario6})\n")

# Bucket 5 (Issuers C & D)
b5_data = net_df[net_df['Bucket'] == 5]
kb5_plus, kb5_minus, kb5_final, scenario5 = calculate_bucket_capital(b5_data, curvature_rho_kl_medium)
print("Bucket 5 (Issuers C & D):")
print(f"  - K_b+ = {kb5_plus:,.0f}")
print(f"  - K_b- = {kb5_minus:,.0f}")
print(f"  - Final K_b = {kb5_final:,.0f} (Selected Scenario: {scenario5})")
print("\n" + "="*50 + "\n")


# --- Step 6: Determine Bucket Sums (Medium Scenario) ---

def calculate_bucket_sum(cvr_data, scenario):
    """Calculates Sb based on the selected scenario."""
    if scenario == "Upward":
        return cvr_data['CVR+'].sum()
    else:
        return cvr_data['CVR-'].sum()

s6 = calculate_bucket_sum(b6_data, scenario6)
s5 = calculate_bucket_sum(b5_data, scenario5)

print("--- Step 6: Bucket Sums (Medium Scenario) ---")
print(f"Bucket 6 Sum (S_b): {s6:,.0f} (from {scenario6} scenario)")
print(f"Bucket 5 Sum (S_b): {s5:,.0f} (from {scenario5} scenario)")
print("\n" + "="*50 + "\n")


# --- Step 7: Cross-Bucket Capital (Medium Scenario) ---

def calculate_rccr(bucket_capitals, bucket_sums, cross_bucket_corr):
    """Calculates the final Risk Class Curvature Requirement (RCCR)."""

    def psi(x, y):
        return 0 if x < 0 and y < 0 else 1

    sum_kb_sq = np.sum(np.array(bucket_capitals)**2)

    correlation_term = 0
    if len(bucket_sums) > 1:
        # Simplified for 2 buckets
        correlation_term = 2 * cross_bucket_corr * bucket_sums[0] * bucket_sums[1] * psi(bucket_sums[0], bucket_sums[1])

    rccr = np.sqrt(max(0, sum_kb_sq + correlation_term))
    return rccr

rccr_medium = calculate_rccr([kb5_final, kb6_final], [s5, s6], curvature_gamma_bc_medium)

print("--- Step 7: Cross-Bucket Capital (Medium Scenario) ---")
print(f"Total Capital (RCCR) for Medium Scenario: £{rccr_medium:,.0f}")
print("\n" + "="*50 + "\n")


# --- Step 8: All Scenarios and Final Capital ---

print("--- Step 8: All Scenarios and Final Capital ---")

scenarios = {
    "High": 1.25,
    "Medium": 1.0,
    "Low": 0.75 # Placeholder, will be calculated with max function
}

results = []

for name, multiplier in scenarios.items():
    # Correct Method: Apply scenario adjustments to the MEDIUM CURVATURE correlations
    rho_curv_scen = curvature_rho_kl_medium
    gamma_curv_scen = curvature_gamma_bc_medium

    if name == "High":
        rho_curv_scen *= multiplier
        gamma_curv_scen *= multiplier
    elif name == "Low":
        rho_curv_scen = max(2 * curvature_rho_kl_medium - 1, 0.75 * curvature_rho_kl_medium)
        gamma_curv_scen = max(2 * curvature_gamma_bc_medium - 1, 0.75 * curvature_gamma_bc_medium)

    # Recalculate Bucket capitals with new intra-bucket correlation
    _, _, kb6_scen, scen6_scen = calculate_bucket_capital(b6_data, rho_curv_scen)
    _, _, kb5_scen, scen5_scen = calculate_bucket_capital(b5_data, rho_curv_scen)

    # Recalculate Bucket sums based on potentially new scenario selections
    s6_scen = calculate_bucket_sum(b6_data, scen6_scen)
    s5_scen = calculate_bucket_sum(b5_data, scen5_scen)

    # Recalculate total RCCR
    rccr_scen = calculate_rccr([kb5_scen, kb6_scen], [s5_scen, s6_scen], gamma_curv_scen)

    results.append({
        "Scenario": name,
        "Intra-Bucket Curv. Corr.": rho_curv_scen,
        "Cross-Bucket Curv. Corr.": gamma_curv_scen,
        "Total Capital (RCCR)": rccr_scen
    })

results_df = pd.DataFrame(results).sort_values("Scenario", key=lambda x: x.map({"Low":0, "Medium":1, "High":2})).set_index("Scenario")

print("Summary of Capital Across All Scenarios:")
print(results_df.to_string(formatters={
    'Intra-Bucket Curv. Corr.': '{:,.2%}'.format,
    'Cross-Bucket Curv. Corr.': '{:,.2%}'.format,
    'Total Capital (RCCR)': '£{:,.0f}'.format
}))

final_capital = results_df['Total Capital (RCCR)'].max()

print("\n" + "-"*50)
print(f"Final Equity Curvature Capital Requirement: £{final_capital:,.0f}")
print("-" * 50)

--- Step 2: Gross Positions ---
   Bucket    Issuer   CVR+   CVR-
0       6  Issuer A  -6447   8842
1       6  Issuer B -49220  48581
2       6  Issuer B   6200 -48309
3       5  Issuer C   8990 -72303
4       5  Issuer D   3638 -28347


--- Step 3: Net Positions ---
Netting the two 'Issuer B' positions as they belong to the same risk factor.
   Bucket    Issuer   CVR+   CVR-
0       5  Issuer C   8990 -72303
1       5  Issuer D   3638 -28347
2       6  Issuer A  -6447   8842
3       6  Issuer B -43020    272


--- Step 4: Correlation Parameters (Medium Scenario) ---
Intra-Bucket Delta Correlation (rho_kl): 25.00%
Intra-Bucket Curvature Correlation (rho_kl^2): 6.25%

Cross-Bucket Delta Correlation (gamma_bc): 15.00%
Cross-Bucket Curvature Correlation (gamma_bc^2): 2.25%


--- Step 5: Bucket-Level Capital (Medium Scenario) ---
Bucket 6 (Issuers A & B):
  - K_b+ = 0
  - K_b- = 8,863
  - Final K_b = 8,863 (Selected Scenario: Downward)

Bucket 5 (Issuers C & D):
  - K_b+ = 9,907
  - K_b- =